#  Multi-label Classification

**다중 분류 vs 다중 레이블 분류**

| 구분    | 다중 분류 (Multi-class)          | 다중 레이블 분류 (Multi-label)         |
| ----- | ---------------------------- | ------------------------------- |
| 정의    | 하나의 샘플이 여러 클래스 중 **하나**에만 속함 | 하나의 샘플이 **여러 클래스에 동시에** 속할 수 있음 |
| 예시    | 고양이, 개, 새 중 하나               | 영화가 Action + Sci-Fi + Drama     |
| 출력 형태 | 정수 인덱스 (`y=3`)               | 이진 벡터 (`y=[1, 0, 1, 0, 1]`)     |
| 모델 출력 | `argmax` 사용                  | `sigmoid` 후 **각 클래스마다 이진 판단**   |

**다중 레이블 문제의 대표 예시**

* 텍스트 분류 (뉴스 → 여러 주제)
* 영화/음악 장르 분류
* 이미지에서 객체 감지 (여러 객체 포함 가능)
* 질병 진단 (동시 복합 질병)

## MultiLabelBinarizer

In [8]:
import pandas as pd  # 데이터프레임 처리

data = pd.DataFrame({  # 영화 줄거리와 다중 장르 라벨을 가진 데이터 생성
    'plot': [
        "A man fights crime in a futuristic city.",        # 영화 1 줄거리
        "A love story set in wartime.",                    # 영화 2 줄거리
        "Aliens invade Earth and a war begins.",           # 영화 3 줄거리
        "A detective solves a complicated crime case.",    # 영화 4 줄거리
        "A dramatic romance in the midst of a tragedy."    # 영화 5 줄거리
    ],
    'genres': [
        ['Action', 'Sci-Fi'],              # 영화 1 장르(다중 라벨)
        ['Romance', 'Drama'],              # 영화 2 장르(다중 라벨)
        ['Action', 'Sci-Fi', 'War'],       # 영화 3 장르(다중 라벨)
        ['Crime', 'Mystery'],              # 영화 4 장르(다중 라벨)
        ['Drama', 'Romance']               # 영화 5 장르(다중 라벨)
    ]
})
data  # 데이터 확인

,plot,genres
0,A man fights crime in a futuristic city.,"[Action, Sci-Fi]"
1,A love story set in wartime.,"[Romance, Drama]"
2,Aliens invade Earth and a war begins.,"[Action, Sci-Fi, War]"
3,A detective solves a complicated crime case.,"[Crime, Mystery]"
4,A dramatic romance in the midst of a tragedy.,"[Drama, Romance]"


In [9]:
data.info

<bound method DataFrame.info of                                             plot                 genres
0       A man fights crime in a futuristic city.       [Action, Sci-Fi]
1                   A love story set in wartime.       [Romance, Drama]
2          Aliens invade Earth and a war begins.  [Action, Sci-Fi, War]
3   A detective solves a complicated crime case.       [Crime, Mystery]
4  A dramatic romance in the midst of a tragedy.       [Drama, Romance]>

In [10]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(data['genres'])
print(y)
print(mlb.classes_)


label_df = pd.DataFrame(y, columns = mlb.classes_, index = data['plot'])
label_df

[[1 0 0 0 0 1 0]
 [0 0 1 0 1 0 0]
 [1 0 0 0 0 1 1]
 [0 1 0 1 0 0 0]
 [0 0 1 0 1 0 0]]
['Action' 'Crime' 'Drama' 'Mystery' 'Romance' 'Sci-Fi' 'War']


,Action,Crime,Drama,Mystery,Romance,Sci-Fi,War
plot,,,,,,,
A man fights crime in a futuristic city.,1,0,0,0,0,1,0
A love story set in wartime.,0,0,1,0,1,0,0
Aliens invade Earth and a war begins.,1,0,0,0,0,1,1
A detective solves a complicated crime case.,0,1,0,1,0,0,0
A dramatic romance in the midst of a tragedy.,0,0,1,0,1,0,0


# 다중레이블 분류 모델

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(data['plot'])

input_df = pd.DataFrame(
    X.toarray(),
    columns = vectorizer.get_feature_names_out(),
    index = data['plot']
)
input_df

,aliens,and,begins,case,city,complicated,crime,detective,dramatic,earth,...,midst,of,romance,set,solves,story,the,tragedy,war,wartime
plot,,,,,,,,,,,,,,,,,,,,,
A man fights crime in a futuristic city.,0.000000,0.000000,0.000000,0.000000,0.442832,0.000000,0.357274,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
A love story set in wartime.,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.474125,0.000000,0.474125,0.000000,0.000000,0.000000,0.474125
Aliens invade Earth and a war begins.,0.408248,0.408248,0.408248,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.408248,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.408248,0.000000
A detective solves a complicated crime case.,0.000000,0.000000,0.000000,0.463693,0.000000,0.463693,0.374105,0.463693,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.463693,0.000000,0.000000,0.000000,0.000000,0.000000
A dramatic romance in the midst of a tragedy.,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.393795,0.000000,...,0.393795,0.393795,0.393795,0.000000,0.000000,0.000000,0.393795,0.393795,0.000000,0.000000


In [12]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

clf = OneVsRestClassifier(LogisticRegression())
clf.fit(X, y)

print('학습 완료')

학습 완료


In [ ]:
test_plot = ['An alien spaceship lands in the middle of a war.']

X_test = vectorizer.transform(test_plot)  # TF-IDF 학습기준으로 벡터화
y_pred = clf.predict(X_test)              # 멀티라벨 예측 (기본 임계값 0.5 기준)
print(y_pred)

y_pred_proba = clf.predict_proba(X_test) # 장르별 예측 확률
print(y_pred_proba)

y_pred = (y_pred_proba >= 0.3).astype(int) # 임계값 조정 (0.3)
print(y_pred)
 
y_pred_label = mlb.inverse_transform(y_pred) # 멀티핫 -> 장르 라벨 리스트 역변환
y_pred_label

[[0 0 0 0 0 0 0]]
[[0.38623748 0.17121135 0.44411123 0.17121135 0.44411123 0.38623748
  0.20204508]]
[[1 0 1 0 1 1 0]]


[('Action', 'Drama', 'Romance', 'Sci-Fi')]

RNN 기반 다중레이블 분류

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer         # 텍스트 -> 정수 시퀀스
from tensorflow.keras.preprocessing.sequence import pad_sequences # 시퀀스 길이 패딩/자르기
import torch

tokenizer = Tokenizer(oov_token = 'OOV')      # 사전에 없는단어 OOV
tokenizer.fit_on_texts(data['plot'])          # 줄거리로 단어 사전 학습(생성)
X = tokenizer.texts_to_sequences(data['plot']) # 줄거리를 정수 시퀀스 형태로 변환
X = pad_sequences(X, maxlen = 10)              # 길이 10으로 패딩처리 
X = torch.tensor(X, dtype = torch.long)        # 임베딩 입력용 LongTensor로 변환
X 

tensor([[ 0,  0,  2,  5,  6,  4,  3,  2,  7,  8],
        [ 0,  0,  0,  0,  2,  9, 10, 11,  3, 12],
        [ 0,  0,  0, 13, 14, 15, 16,  2, 17, 18],
        [ 0,  0,  0,  2, 19, 20,  2, 21,  4, 22],
        [ 0,  2, 23, 24,  3, 25, 26, 27,  2, 28]])

In [ ]:
mlb = MultiLabelBinarizer()                # 다중 라벨 -> 멀티핫 변환기
y = mlb.fit_transform(data['genres'])      # 장르 리스트 -> 멀티핫 행렬
y = torch.tensor(y, dtype = torch.float)   # BCE 계열 손실 입력값 -> FloatTensor
y

In [20]:
import torch
import torch.nn as nn

class GRUSentimentNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )
        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.embedding(x)       # (B, T) -> (B, T, E)
        _, hidden = self.gru(x)     # hidden: (L, B, H)
        logits = self.fc(hidden[-1])  # (B, output_dim)
        return logits


vocab_size = len(tokenizer.word_index) + 1

model = GRUSentimentNet(
    vocab_size=vocab_size,
    embedding_dim=100,
    hidden_dim=64,
    output_dim=len(mlb.classes_)
)

print(model)

GRUSentimentNet(
  (embedding): Embedding(29, 100, padding_idx=0)
  (gru): GRU(100, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=7, bias=True)
)


In [22]:
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.optim as optim

# NumPy 배열이라면 Tensor로 변환
X = torch.as_tensor(X, dtype=torch.long)    # (데이터 수, 문장 길이)
y = torch.as_tensor(y, dtype=torch.float32) # (데이터 수, 클래스 수)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 100

for epoch in tqdm(range(epochs)):
    model.train()

    optimizer.zero_grad()

    logits = model(X)
    loss = criterion(logits, y)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch + 1}/{epochs}, "
            f"Loss: {loss.item():.4f}"
        )

  0%|          | 0/100 [00:00<?, ?it/s]

Epoch 10/100, Loss: 0.4989
Epoch 20/100, Loss: 0.3229
Epoch 30/100, Loss: 0.2101
Epoch 40/100, Loss: 0.1434
Epoch 50/100, Loss: 0.1046
Epoch 60/100, Loss: 0.0805
Epoch 70/100, Loss: 0.0644
Epoch 80/100, Loss: 0.0531
Epoch 90/100, Loss: 0.0448
Epoch 100/100, Loss: 0.0385


In [ ]:
test_plot = ["Anm alien spaceship lands in the middle of a war."]

X_test = tokenizer.texts_to_sequences(test_plot)  # 정수 시퀀스 변환
X_test = pad_sequences(X_test, maxlen = 10)       # 학습과 동일한 maxlen으로 패딩
X_test = torch.tensor(X_test, dtype = torch.long) # 임베딩 학습용 LongTensor

model.eval()
with torch.no_grad():
    output = model(X_test)
    p = torch.sigmoid(output)
    pred = (p >= 0.5).int()
    pred_label = mlb.inverse_transform(pred)      # 멀티핫 -> 라벨 리스트
    print(pred_label)

[('Action', 'Sci-Fi')]


# Bert Tokenizer/ Embedding 적용

In [25]:
%pip install transformers huggingface_hub

  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   ----------------- ---------------------- 5.2/11.7 MB 35.3 MB/s eta 0:00:01
   ---------------------------------------- 11.7/11.7 MB 38.7 MB/s  0:00:00
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl (355 kB)

  Attempting uninstall: tokenizers

    Found existing installation: tokenizers 0.23.1

    Uninstalling tokenizers-0.23.1:

      Successfully uninstalled tokenizers-0.23.1

   ------------- -------------------------- 1/3 [tokenizers]
   ------------- -------------------------- 1/3 [tokenizers]
   ------------- -------------------------- 1/3 [tokenizers]
   -------------------------- ------------- 2/3 [transformers]
   -

In [ ]:
# BERT 사전 학습된 tokenizer / model 가져오기
from transformers import BertTokenizer, BertModel

model_name = 'bert-base-uncased' # 소문자 영어 기반 Bert 이름
bert_tokenizer = BertTokenizer.from_pretrained(model_name) # 사전학습 토크나이저
bert_model = BertModel.from_pretrained(model_name)         # 사전학습 Bert 모델


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

c:\Users\SJ\OneDrive\Desktop\study\NLP\nlp_venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SJ\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# BERT 임베딩 사전학습 차원수 : 768
import torch

# 여러 문장을 BERT에 넣어 토큰 단위 문맥 임베딩을 반환
def get_bert_embedding(plots):

    # 토큰화 + 패딩 + 텐서 변환
    encoded = bert_tokenizer(plots, padding = True, truncation = True, return_tensors = 'pt')

    # 임베딩 추출만 할 것이므로 기울기 계산 비활성화
    with torch.no_grad():
        output = bert_model(**encoded) # encoded : input_ids/token_type_ids/attention_mask

    return output.last_hidden_state # (batch_size, seq_len, 768) # 토큰별 임베딩 반환

plots = data['plot'].values.tolist() # 줄거리 리스트
X_tensor = get_bert_embedding(plots) # Bert 임베딩 생성
print(X_tensor.shape) # (Batch_size, seq_len, embedding_dim)

torch.Size([5, 12, 768])


In [31]:
mlb = MultiLabelBinarizer()                # 다중 라벨 -> 멀티핫 변환기
y = mlb.fit_transform(data['genres'])      # 장르 리스트 -> 멀티핫 행렬
y_tensor = torch.tensor(y, dtype = torch.float)   # BCE 계열 손실 입력값 -> FloatTensor
y_tensor

tensor([[1., 0., 0., 0., 0., 1., 0.],
        [0., 0., 1., 0., 1., 0., 0.],
        [1., 0., 0., 0., 0., 1., 1.],
        [0., 1., 0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 1., 0., 0.]])

In [33]:
import torch
import torch.nn as nn

class MultiLabelNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x: (batch_size, sequence_length, input_dim)
        _, hidden = self.gru(x)
        logits = self.fc(hidden[-1])  # (batch_size, output_dim)
        return logits


input_dim = X_tensor.shape[-1]
hidden_dim = 64
output_dim = y_tensor.shape[-1]

model = MultiLabelNet(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    output_dim=output_dim
)

print(model)

MultiLabelNet(
  (gru): GRU(768, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=7, bias=True)
)


In [36]:
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# BERT에서 미리 만든 임베딩과 다중 레이블 정답으로 미니배치 구성
train_dataset = TensorDataset(X_tensor.float(), y_tensor.float())
train_dataloader = DataLoader(train_dataset, batch_size=2, shuffle=True)

model = model.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 100

# 학습
for epoch in tqdm(range(epochs)):
    total_loss, correct, total = 0, 0, 0

    model.train()  # 학습 모드
    for inputs, labels in train_dataloader:
        inputs = inputs.to(device).float()
        labels = labels.to(device).float()

        optimizer.zero_grad()

        output = model(inputs)
        loss = criterion(output, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.detach().cpu().item()
        p = torch.sigmoid(output)
        pred = (p >= 0.5).float()

        correct += (pred == labels).sum().float().detach().cpu().item()
        total += labels.numel()

    train_loss = total_loss / len(train_dataloader)
    train_accuracy = correct / total

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch + 1}/{epochs} "
            f"Loss : {train_loss:.4f} "
            f"Accuracy : {train_accuracy:.4f}"
        )

  0%|          | 0/100 [00:00<?, ?it/s]

Epoch 10/100 Loss : 0.2529 Accuracy : 1.0000
Epoch 20/100 Loss : 0.1337 Accuracy : 1.0000
Epoch 30/100 Loss : 0.0881 Accuracy : 1.0000
Epoch 40/100 Loss : 0.0624 Accuracy : 1.0000
Epoch 50/100 Loss : 0.0470 Accuracy : 1.0000
Epoch 60/100 Loss : 0.0395 Accuracy : 1.0000
Epoch 70/100 Loss : 0.0321 Accuracy : 1.0000
Epoch 80/100 Loss : 0.0267 Accuracy : 1.0000
Epoch 90/100 Loss : 0.0208 Accuracy : 1.0000
Epoch 100/100 Loss : 0.0187 Accuracy : 1.0000


In [37]:
test_plot = [
    "An alien spaceship lands in the middle of a war.",
    "A young couple falls in love during a tragic event.",
    "A detective investigates a mysterious murder case."
]

X_test = get_bert_embedding(test_plot).to(device)

model.eval()
with torch.no_grad():
    output = model(X_test)
    probabilities = torch.sigmoid(output)
    predictions = (probabilities >= 0.5).int().cpu().numpy()
    predicted_labels = mlb.inverse_transform(predictions)

for plot, labels in zip(test_plot, predicted_labels):
    print(plot)
    print(labels)

An alien spaceship lands in the middle of a war.
('Action', 'Sci-Fi')
A young couple falls in love during a tragic event.
('Drama', 'Sci-Fi')
A detective investigates a mysterious murder case.
('Crime', 'Mystery')
